<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/21-evaluation-interpretability-robustness-responsibility.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Evaluation, Interpretability, Robustness, and Responsible Deep Learning** {#evaluation-interpretability-robustness-responsibility}

Evaluation is an argument supported by evidence: a particular artifact, on defined data and operating conditions, is suitable for a stated decision. An average benchmark score is only one premise. Reliable use also requires calibrated uncertainty, known failure boundaries, stress tests, subgroup evidence, data lineage, privacy analysis, and a response when assumptions fail. Interpretability supports this argument when it tests a concrete hypothesis; an attractive heatmap alone does not establish trust.

The chapter uses scikit-learn's copy of the [UCI Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49) dataset, licensed **CC BY 4.0**. Digits 0 through 8 define a nine-class in-distribution (ID) task. Every example of digit 9 is excluded from model fitting and retained as a semantic out-of-distribution (OOD) set. One deterministic split and one small CNN support all later calibration, probing, attribution, adversarial, privacy, and subgroup experiments. This controlled construction makes mechanisms comparable, but digit 9 is only one easy OOD family and the dataset contains no demographic attributes.

![Digits zero through eight form the ID task while digit nine is reserved as OOD.](assets/dl21-id-ood-grid.png){fig-align="center" width="68%" fig-alt="Ten small handwritten digit images are shown; labels zero through eight are in-distribution and label nine is marked out-of-distribution."}

*Data source: Alpaydin and Kaynak, [UCI Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49), CC BY 4.0. Samples are displayed from scikit-learn's documented `load_digits` copy.*

<details>
<summary><strong>PyTorch: establish the shared ID/OOD evaluation workload</strong></summary>

```python
import hashlib
import io
import json
import random

import numpy as np
import sklearn
import torch
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=2121):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_ids = np.arange(len(digits.data))
id_ids = all_ids[digits.target != 9]
ood_ids = all_ids[digits.target == 9]
train_ids, remaining_ids = train_test_split(
    id_ids, test_size=0.30, stratify=digits.target[id_ids], random_state=2121
)
val_ids, test_ids = train_test_split(
    remaining_ids,
    test_size=0.50,
    stratify=digits.target[remaining_ids],
    random_state=2121,
)

# UCI fixes intensities to 0..16; division by 16 does not fit a test statistic.
images = torch.tensor(digits.images / 16.0, dtype=torch.float32)[:, None, :, :]
targets = torch.tensor(digits.target, dtype=torch.long)
train_dataset = TensorDataset(images[train_ids], targets[train_ids])
val_dataset = TensorDataset(images[val_ids], targets[val_ids])
test_dataset = TensorDataset(images[test_ids], targets[test_ids])
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(2121),
)
val_loader = DataLoader(val_dataset, batch_size=128)
test_loader = DataLoader(test_dataset, batch_size=128)


class TinyDigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 4 * 4, 64)
        self.dropout = nn.Dropout(0.15)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 9)

    def forward_intermediates(self, x):
        conv1 = F.gelu(self.conv1(x))
        conv2 = F.gelu(self.conv2(conv1))
        pooled = F.avg_pool2d(conv2, kernel_size=2).flatten(1)
        embedding = F.gelu(self.fc1(pooled))
        hidden = F.gelu(self.fc2(self.dropout(embedding)))
        logits = self.fc3(hidden)
        return logits, {"conv1": conv1, "conv2": conv2, "embedding": embedding, "hidden": hidden}

    def logits_from_embedding(self, embedding):
        hidden = F.gelu(self.fc2(self.dropout(embedding)))
        return self.fc3(hidden)

    def forward(self, x):
        return self.forward_intermediates(x)[0]


def batched_logits(model, inputs, batch_size=128):
    model.eval()
    outputs = []
    with torch.inference_mode():
        for start in range(0, len(inputs), batch_size):
            outputs.append(model(inputs[start : start + batch_size]))
    return torch.cat(outputs)


seed_everything()
model = TinyDigitCNN()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
for _ in range(18):
    model.train()
    for x, y in train_loader:
        loss = F.cross_entropy(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

id_test_x, id_test_y = images[test_ids], targets[test_ids]
ood_x = images[ood_ids]
id_test_logits = batched_logits(model, id_test_x)
baseline_accuracy = float(id_test_logits.argmax(1).eq(id_test_y).float().mean())
assert not set(train_ids) & set(test_ids)
assert set(targets[train_ids].tolist()) == set(range(9)) and set(targets[ood_ids].tolist()) == {9}
assert baseline_accuracy > 0.93
print({"split": (len(train_ids), len(val_ids), len(test_ids)), "OOD": len(ood_ids), "ID_accuracy": round(baseline_accuracy, 3)})
```

</details>

The model is deliberately small and the OOD boundary is deliberately explicit. Results validate evaluation procedures on this workload; they do not establish readiness for handwriting recognition in the wild, demographic fairness, privacy guarantees, or adversarial security.


### **Evaluation Across Modalities and Tasks** {#evaluation-across-modalities-tasks}

A metric is meaningful only relative to a task, unit of analysis, data distribution, and decision cost. Classification may require accuracy, macro-F1, per-class recall, calibration, and abstention behavior. Retrieval needs ranking metrics such as recall@(k), MRR, or nDCG and must define the relevance set. Segmentation reports IoU/Dice at object or pixel level. Generation needs multiple references or human judgments because lexical overlap, factuality, diversity, and preference are not the same property. Representation learning needs downstream probes or transfer protocols, while reinforcement learning requires return distributions, constraint violations, and evaluation across seeds.

![An evaluation stack from unit tests to system and decision context.](assets/dl21-evaluation-stack.svg){fig-align="center" width="76%" fig-alt="Five evaluation layers progress from unit tests through dataset, slice, stress, and system evidence, all grounded in a decision context."}

The evaluation unit must prevent leakage. Split by patient rather than scan, speaker rather than utterance, user rather than interaction, and time when the future is the deployment target. Freeze preprocessing and thresholds before touching the test set. Repeated tuning on a public benchmark converts it into training feedback even if its rows never enter gradient descent.

![Confusion matrix for the held-out in-distribution Digits test set.](assets/dl21-confusion-matrix.png){fig-align="center" width="62%" fig-alt="A nine by nine confusion matrix shows counts for true and predicted digit classes zero through eight."}

<details>
<summary><strong>Python: combine aggregate, class, and selective-prediction evidence</strong></summary>

```python
id_predictions = id_test_logits.argmax(dim=1)
id_probabilities = id_test_logits.softmax(dim=1)
id_confidence = id_probabilities.max(dim=1).values
macro_f1 = f1_score(id_test_y.numpy(), id_predictions.numpy(), average="macro")
matrix = confusion_matrix(id_test_y.numpy(), id_predictions.numpy(), labels=list(range(9)))
per_class_recall = matrix.diagonal() / matrix.sum(axis=1).clip(min=1)

# Selective prediction: abstain on the lowest-confidence 20 percent.
keep = id_confidence >= torch.quantile(id_confidence, 0.20)
selective_accuracy = float(id_predictions[keep].eq(id_test_y[keep]).float().mean())
coverage = float(keep.float().mean())
assert matrix.sum() == len(test_ids) and 0 < coverage <= 1
print({
    "accuracy": round(baseline_accuracy, 3),
    "macro_F1": round(macro_f1, 3),
    "worst_class_recall": round(float(per_class_recall.min()), 3),
    "selective_accuracy": round(selective_accuracy, 3),
    "coverage": round(coverage, 3),
})
```

</details>

Selective accuracy is not free improvement: abstained cases move to another workflow whose cost and quality must be measured. Report uncertainty intervals or multiple seeds when sampling or optimization variance is material, and preserve raw predictions so later slice analyses do not depend on rounded summary numbers.


### **Calibration and Predictive Uncertainty** {#calibration-predictive-uncertainty}

Accuracy asks whether the top class is correct; calibration asks whether confidence corresponds to empirical frequency. A calibrated set of predictions made with confidence 0.8 should be correct about 80% of the time. Negative log-likelihood and Brier score are proper scoring rules that evaluate the full distribution. Expected calibration error (ECE) bins predictions and computes

$$\mathrm{ECE}=\sum_{m=1}^{M}\frac{|B_m|}{n}\left|\operatorname{acc}(B_m)-\operatorname{conf}(B_m)\right|.$$

ECE is interpretable but depends on binning and can hide class-conditional errors. Reliability diagrams retain more structure. Temperature scaling learns one positive scalar (T) on validation logits and predicts (\operatorname{softmax}(z/T)); it changes confidence without changing the argmax. The [calibration study by Guo et al.](https://proceedings.mlr.press/v70/guo17a.html) established it as a strong post-hoc baseline.

![Reliability diagram before and after temperature scaling.](assets/dl21-calibration.png){fig-align="center" width="64%" fig-alt="A reliability chart compares raw and temperature-scaled confidence against empirical accuracy and includes a perfect-calibration diagonal."}

Predictive uncertainty has different sources. Aleatoric uncertainty reflects irreducible ambiguity in the observation; epistemic uncertainty reflects uncertainty about model parameters or unsupported regions. Softmax entropy mixes several effects and is not automatically epistemic. Ensembles and approximate Bayesian methods can expose model disagreement, but only under their training and approximation assumptions.

<details>
<summary><strong>PyTorch: fit temperature scaling and compare it with MC-dropout disagreement</strong></summary>

```python
def expected_calibration_error(logits, labels, bins=10):
    probabilities = logits.softmax(dim=1)
    confidence, prediction = probabilities.max(dim=1)
    correctness = prediction.eq(labels).float()
    edges = torch.linspace(0, 1, bins + 1)
    ece = torch.tensor(0.0)
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (confidence > lower) & (confidence <= upper)
        if mask.any():
            ece += mask.float().mean() * (correctness[mask].mean() - confidence[mask].mean()).abs()
    return float(ece)


validation_logits = batched_logits(model, images[val_ids])
validation_labels = targets[val_ids]
temperature_grid = torch.linspace(0.5, 4.0, 141)
validation_nll = torch.tensor([
    F.cross_entropy(validation_logits / value, validation_labels) for value in temperature_grid
])
temperature = float(temperature_grid[validation_nll.argmin()])
raw_ece = expected_calibration_error(id_test_logits, id_test_y)
calibrated_ece = expected_calibration_error(id_test_logits / temperature, id_test_y)
raw_test_nll = float(F.cross_entropy(id_test_logits, id_test_y))
calibrated_test_nll = float(F.cross_entropy(id_test_logits / temperature, id_test_y))

# Enable only Dropout at test time; this model has no BatchNorm state to disturb.
model.train()
with torch.inference_mode():
    mc_probabilities = torch.stack([model(id_test_x[:64]).softmax(1) for _ in range(24)])
model.eval()
predictive_entropy = -(mc_probabilities.mean(0) * mc_probabilities.mean(0).clamp_min(1e-8).log()).sum(1)
disagreement = mc_probabilities.var(0).sum(1)
assert temperature > 0 and predictive_entropy.shape == disagreement.shape == (64,)
print({
    "temperature": round(temperature, 3),
    "ECE_raw/calibrated": (round(raw_ece, 3), round(calibrated_ece, 3)),
    "NLL_raw/calibrated": (round(raw_test_nll, 3), round(calibrated_test_nll, 3)),
    "mean_MC_disagreement": round(float(disagreement.mean()), 5),
})
```

</details>

Calibration must be rechecked after domain, class-prior, precision, or model changes. A globally calibrated model can remain miscalibrated on a critical subgroup. Choose an uncertainty method according to the decision it drives: abstention, active learning, OOD routing, or risk-sensitive ranking require different validation evidence.


### **Out-of-Distribution Detection** {#out-of-distribution-detection}

OOD is defined relative to the training distribution and intended task. A new background style is covariate shift; an unseen class such as digit 9 is semantic shift; a changed label rule is concept shift. Detection, generalization, and adaptation are separate goals: a detector may flag an input without knowing how to classify it, while a robust model may remain accurate under a shift it never explicitly detects.

Maximum softmax probability (MSP) is a useful baseline: low maximum confidence suggests OOD, but discriminative networks can be confident far from training data. The energy score for logits (z_k(x)) and temperature (T) is

$$E(x)=-T\log\sum_{k=1}^{K}\exp(z_k(x)/T).$$

ID inputs often have lower energy. The [energy-based OOD work](https://proceedings.neurips.cc/paper/2020/hash/f5496252609c43eb8a3d147ab9b9c006-Abstract.html) formalizes this alternative to MSP. Neither score is a probability that an input is OOD.

![Energy-score distributions for ID digits and held-out digit nine.](assets/dl21-ood-energy.png){fig-align="center" width="66%" fig-alt="Overlapping histograms compare lower energy scores for in-distribution digits zero through eight with scores for held-out digit nine."}

<details>
<summary><strong>PyTorch: compare MSP and energy detection on the held-out digit</strong></summary>

```python
ood_logits = batched_logits(model, ood_x)
id_msp_score = 1.0 - id_test_logits.softmax(1).max(1).values
ood_msp_score = 1.0 - ood_logits.softmax(1).max(1).values
id_energy = -torch.logsumexp(id_test_logits, dim=1)
ood_energy = -torch.logsumexp(ood_logits, dim=1)


def ood_auc(id_scores, shifted_scores):
    labels = np.concatenate([np.zeros(len(id_scores)), np.ones(len(shifted_scores))])
    scores = torch.cat([id_scores, shifted_scores]).numpy()
    return roc_auc_score(labels, scores)


msp_ood_auc = ood_auc(id_msp_score, ood_msp_score)
energy_ood_auc = ood_auc(id_energy, ood_energy)
validation_energy = -torch.logsumexp(validation_logits, dim=1)
ood_energy_threshold = float(torch.quantile(validation_energy, 0.95))
id_false_positive_rate = float((id_energy > ood_energy_threshold).float().mean())
ood_true_positive_rate = float((ood_energy > ood_energy_threshold).float().mean())
assert 0 <= msp_ood_auc <= 1 and 0 <= energy_ood_auc <= 1
print({
    "MSP_AUROC": round(msp_ood_auc, 3),
    "energy_AUROC": round(energy_ood_auc, 3),
    "validation_threshold_ID_FPR": round(id_false_positive_rate, 3),
    "digit9_TPR": round(ood_true_positive_rate, 3),
})
```

</details>

AUROC is threshold-independent but may look favorable under unrealistic class balance. Also report FPR at a required TPR, precision-recall curves, and workload prevalence. Test multiple near-OOD and far-OOD families; a detector tuned on digit 9 may fail on blur, blank inputs, or new acquisition devices. Set thresholds on validation data and preserve an operational action such as abstain, route, or collect for review.


### **Feature and Representation Probing** {#feature-representation-probing}

A probe freezes a representation and trains a controlled readout for a property of interest. It asks whether information is **decodable** from that representation under the probe's capacity and data budget. Comparing layers can reveal where class, syntax, geometry, or another property becomes linearly accessible. It does not prove that the original model uses the information or that the decoded direction is causal.

![A frozen network feeding controlled probes with explicit claim boundaries.](assets/dl21-probing.svg){fig-align="center" width="74%" fig-alt="Representations from several frozen layers feed the same controlled linear probe; a final box warns that decodability is not causality."}

Probe design must control data splits, class balance, dimensionality, regularization, and probe capacity. A high-capacity nonlinear probe may learn the task from weak traces. A high-dimensional random representation can be surprisingly separable. Selectivity baselines compare the true property with randomized labels or matched control tasks.

<details>
<summary><strong>PyTorch and scikit-learn: compare linear decodability across CNN layers</strong></summary>

```python
def collect_representations(model, inputs, batch_size=128):
    model.eval()
    storage = {"pixels": [], "conv1": [], "conv2": [], "embedding": []}
    with torch.inference_mode():
        for start in range(0, len(inputs), batch_size):
            batch = inputs[start : start + batch_size]
            _, parts = model.forward_intermediates(batch)
            storage["pixels"].append(batch.flatten(1))
            storage["conv1"].append(F.avg_pool2d(parts["conv1"], 2).flatten(1))
            storage["conv2"].append(F.avg_pool2d(parts["conv2"], 2).flatten(1))
            storage["embedding"].append(parts["embedding"])
    return {name: torch.cat(values).numpy() for name, values in storage.items()}


train_representations = collect_representations(model, images[train_ids])
test_representations = collect_representations(model, id_test_x)
probe_scores = {}
for layer_name in train_representations:
    probe = LogisticRegression(max_iter=1200, C=1.0, random_state=2121)
    probe.fit(train_representations[layer_name], targets[train_ids].numpy())
    probe_scores[layer_name] = probe.score(test_representations[layer_name], id_test_y.numpy())

assert set(probe_scores) == {"pixels", "conv1", "conv2", "embedding"}
print({name: round(score, 3) for name, score in probe_scores.items()})
```

</details>

A layer with a weaker linear score may still contain useful nonlinear information; a stronger score may reflect nuisance correlation. Probe results are most valuable when paired with interventions, transfer tests, and negative controls, not presented as a complete explanation of representation learning.


### **Gradient-Based Attribution** {#gradient-based-attribution}

Input-gradient attribution computes how a selected score changes under an infinitesimal feature change. For target logit (f_c(x)), saliency is often (S_i=|\partial f_c(x)/\partial x_i|). It is local, model-specific, and inexpensive. Saturated nonlinearities can yield small gradients for important features, and tiny input changes can produce visually unstable maps.

SmoothGrad averages gradients from noisy copies, (\bar{S}(x)=\frac{1}{N}\sum_n S(x+\epsilon_n)), reducing visual noise without changing the underlying model. Gradient times input adds the current feature magnitude, but the zero reference may be semantically inappropriate. The target must be explicit: predicted logit, true-class logit, margin, probability, or loss answer different questions.

![Input, raw gradient saliency, and SmoothGrad for the same prediction.](assets/dl21-saliency.png){fig-align="center" width="66%" fig-alt="Three panels show a handwritten digit, its absolute input-gradient map, and a smoother averaged gradient map."}

<details>
<summary><strong>PyTorch: compute saliency and test it with feature deletion</strong></summary>

```python
correct_positions = torch.where(id_predictions.eq(id_test_y))[0]
representative_position = int(correct_positions[id_confidence[correct_positions].argmax()])
representative_x = id_test_x[representative_position : representative_position + 1]
representative_target = int(id_predictions[representative_position])

gradient_input = representative_x.clone().requires_grad_(True)
target_score = model(gradient_input)[0, representative_target]
saliency = torch.autograd.grad(target_score, gradient_input)[0].abs()

noise_generator = torch.Generator().manual_seed(2121)
smooth_gradients = []
for _ in range(32):
    noisy = (representative_x + 0.08 * torch.randn(representative_x.shape, generator=noise_generator)).clamp(0, 1)
    noisy.requires_grad_(True)
    smooth_gradients.append(torch.autograd.grad(model(noisy)[0, representative_target], noisy)[0].abs())
smoothgrad = torch.stack(smooth_gradients).mean(0)

# A deletion test checks whether top-attributed pixels affect the chosen score.
top_pixels = smoothgrad.flatten().topk(10).indices
deleted = representative_x.clone().flatten()
deleted[top_pixels] = 0.0
deleted = deleted.view_as(representative_x)
with torch.inference_mode():
    original_score = float(model(representative_x)[0, representative_target])
    deleted_score = float(model(deleted)[0, representative_target])
assert saliency.shape == smoothgrad.shape == representative_x.shape
print({"target": representative_target, "original_logit": round(original_score, 3), "after_top10_deletion": round(deleted_score, 3)})
```

</details>

A map should be evaluated with sanity checks: randomize model weights or labels, compare multiple baselines and seeds, and measure deletion/insertion effects against random or edge-based controls. Plausibility to a human is not faithfulness to the model's computation.


### **Grad-CAM and Integrated Gradients** {#grad-cam-integrated-gradients}

Grad-CAM localizes a target in a convolutional feature map. For channel (k) with activation (A^k), it averages the target gradient spatially,

$$\alpha_k^c=\frac{1}{Z}\sum_{i,j}\frac{\partial y^c}{\partial A_{ij}^k}, \qquad L^c=\operatorname{ReLU}\left(\sum_k\alpha_k^cA^k\right).$$

The map is coarse because it inherits the feature-map resolution; ReLU keeps evidence that increases the target score. The [Grad-CAM paper](https://openaccess.thecvf.com/content_iccv_2017/html/Selvaraju_Grad-CAM_Visual_Explanations_ICCV_2017_paper.html) emphasizes class-discriminative localization rather than pixel-level causal segmentation.

Integrated Gradients (IG) accumulates gradients along a path from baseline (x') to input (x):

$$\operatorname{IG}_i(x)=(x_i-x_i')\int_0^1\frac{\partial f(x'+\alpha(x-x'))}{\partial x_i}\,d\alpha.$$

It satisfies a completeness relation under suitable conditions: attributions sum to (f(x)-f(x')). The [original IG paper](https://proceedings.mlr.press/v70/sundararajan17a.html) makes baseline choice part of the explanation definition.

![Input image with Grad-CAM and Integrated Gradients explanations.](assets/dl21-gradcam-ig.png){fig-align="center" width="68%" fig-alt="The same digit is shown alongside a coarse Grad-CAM overlay and a signed Integrated Gradients pixel map."}

<details>
<summary><strong>PyTorch: implement Grad-CAM and verify Integrated Gradients completeness</strong></summary>

```python
def grad_cam(model, x, target):
    model.eval()
    input_tensor = x.detach().clone().requires_grad_(True)
    logits, parts = model.forward_intermediates(input_tensor)
    feature_map = parts["conv2"]
    feature_map.retain_grad()
    model.zero_grad()
    logits[0, target].backward()
    channel_weights = feature_map.grad.mean(dim=(2, 3), keepdim=True)
    heatmap = F.relu((channel_weights * feature_map).sum(dim=1, keepdim=True))
    heatmap = F.interpolate(heatmap, size=(8, 8), mode="bilinear", align_corners=False)
    return (heatmap / heatmap.max().clamp_min(1e-8)).detach()


def integrated_gradients(model, x, target, steps=96):
    model.eval()
    baseline = torch.zeros_like(x)
    gradients = []
    for alpha in torch.linspace(0.0, 1.0, steps + 1)[1:]:
        interpolated = (baseline + alpha * (x - baseline)).detach().requires_grad_(True)
        gradients.append(torch.autograd.grad(model(interpolated)[0, target], interpolated)[0])
    attribution = (x - baseline) * torch.stack(gradients).mean(0)
    with torch.inference_mode():
        output_difference = model(x)[0, target] - model(baseline)[0, target]
    completeness_error = float(attribution.sum() - output_difference)
    return attribution.detach(), completeness_error


cam = grad_cam(model, representative_x, representative_target)
integrated, completeness_error = integrated_gradients(model, representative_x, representative_target)
assert cam.shape == integrated.shape == representative_x.shape
assert abs(completeness_error) < 0.20
print({"target": representative_target, "IG_completeness_error": round(completeness_error, 4)})
```

</details>

Grad-CAM asks where a convolutional representation supports a target; IG asks how the target changes along a chosen baseline path. Their maps need not agree. Compare baselines, layers, numerical steps, and perturbation tests, and treat explanation stability as an evaluated property rather than a visual assumption.


### **Mechanistic and Concept-Level Interpretability** {#mechanistic-concept-interpretability}

Feature attributions remain tied to pixels or tokens. Concept methods instead define a human-level property through positive and negative examples, fit a direction in activation space, and ask whether target outputs are sensitive along it. A concept activation vector (CAV) (v_C) enables the directional derivative (S_{C,c}(x)=\nabla_h f_c(h(x))\cdot v_C). TCAV aggregates the fraction of examples with positive sensitivity and repeats the test against random concepts; see the [TCAV paper](https://proceedings.mlr.press/v80/kim18d.html).

![From concept examples to activation directions and causal interventions.](assets/dl21-concepts.svg){fig-align="center" width="74%" fig-alt="Concept examples define a direction in activation space, which is measured by directional derivatives and then followed by causal activation interventions."}

Mechanistic interpretability makes a stronger claim about computation. Ablation removes a component; activation patching replaces internal state from a source example; causal tracing localizes where information changes an output; circuit analysis proposes a sparse set of components and interactions. These interventions can still be off-manifold or redundant, so controls and replication are essential.

<details>
<summary><strong>PyTorch: construct a stroke-density CAV and measure directional sensitivity</strong></summary>

```python
train_embeddings = torch.tensor(train_representations["embedding"], dtype=torch.float32)
train_ink = images[train_ids].flatten(1).sum(1)
low_threshold, high_threshold = torch.quantile(train_ink, torch.tensor([0.25, 0.75]))
concept_mask = (train_ink <= low_threshold) | (train_ink >= high_threshold)
concept_labels = (train_ink[concept_mask] >= high_threshold).long().numpy()
concept_probe = LogisticRegression(max_iter=1000, random_state=2121)
concept_probe.fit(train_embeddings[concept_mask].numpy(), concept_labels)
cav = torch.tensor(concept_probe.coef_[0], dtype=torch.float32)
cav = cav / cav.norm().clamp_min(1e-8)

model.eval()
_, test_parts = model.forward_intermediates(id_test_x)
test_embedding = test_parts["embedding"].detach().requires_grad_(True)
logits_from_embedding = model.logits_from_embedding(test_embedding)
selected_logits = logits_from_embedding.gather(1, id_predictions[:, None]).sum()
embedding_gradient = torch.autograd.grad(selected_logits, test_embedding)[0]
concept_sensitivity = embedding_gradient @ cav
tcav_like_fraction = float((concept_sensitivity > 0).float().mean())
concept_train_accuracy = concept_probe.score(train_embeddings[concept_mask].numpy(), concept_labels)
assert cav.shape == (64,) and 0 <= tcav_like_fraction <= 1
print({"concept_probe_training_accuracy": round(concept_train_accuracy, 3), "positive_sensitivity_fraction": round(tcav_like_fraction, 3)})
```

</details>

This is a teaching-scale TCAV-like calculation, not a validated claim that the network contains a stable “ink” concept. Stroke density correlates with digit class, the concept examples are algorithmically defined, and only one direction is fitted. A serious study repeats concept construction, uses matched random controls and held-out concept data, tests statistical stability, and follows correlation with interventions.


### **Adversarial Examples and Robustness** {#adversarial-examples-robustness}

Robustness asks whether behavior remains acceptable under specified perturbations. Natural corruptions model acquisition changes such as blur, noise, compression, or missing sensors. Adversarial examples optimize an input within a threat-model constraint to increase loss or force a target. Under an (L_\infty) budget (\epsilon), FGSM uses (x'=\operatorname{clip}(x+\epsilon\operatorname{sign}(\nabla_x\mathcal{L}),0,1)); projected gradient descent (PGD) repeats smaller steps and projects back into the allowed set.

![Clean, random-noise, and PGD inputs with their resulting accuracies.](assets/dl21-adversarial.png){fig-align="center" width="76%" fig-alt="A digit is shown clean, with random noise, and after PGD; a bar chart compares accuracy under the same infinity-norm budget."}

An (L_p) ball is mathematically convenient but not a complete perceptual or application threat model. Spatial transforms, patches, prompt injection, poisoning, model extraction, and physical attacks require different capabilities and oracles. The seminal [adversarial examples work](https://arxiv.org/abs/1412.6572) showed that worst-case directions differ qualitatively from ordinary random noise.

<details>
<summary><strong>PyTorch: compare random perturbations with iterative worst-case perturbations</strong></summary>

```python
def pgd_attack(model, x, y, epsilon=0.16, step_size=0.04, steps=6):
    model.eval()
    adversarial = x.detach().clone()
    for _ in range(steps):
        adversarial.requires_grad_(True)
        loss = F.cross_entropy(model(adversarial), y)
        gradient = torch.autograd.grad(loss, adversarial)[0]
        adversarial = adversarial.detach() + step_size * gradient.sign()
        adversarial = torch.max(torch.min(adversarial, x + epsilon), x - epsilon).clamp(0, 1)
    return adversarial.detach()


adversarial_x = pgd_attack(model, id_test_x, id_test_y)
random_generator = torch.Generator().manual_seed(2121)
random_noise = torch.empty(id_test_x.shape).uniform_(-0.16, 0.16, generator=random_generator)
random_x = (id_test_x + random_noise).clamp(0, 1)
with torch.inference_mode():
    random_accuracy = float(model(random_x).argmax(1).eq(id_test_y).float().mean())
    adversarial_accuracy = float(model(adversarial_x).argmax(1).eq(id_test_y).float().mean())
maximum_change = float((adversarial_x - id_test_x).abs().max())
assert maximum_change <= 0.16001 and adversarial_accuracy <= baseline_accuracy
print({"clean": round(baseline_accuracy, 3), "random": round(random_accuracy, 3), "PGD": round(adversarial_accuracy, 3), "L_inf": round(maximum_change, 3)})
```

</details>

Attack strength is part of the result: record norm, budget, steps, restarts, target, white/black-box access, and gradient handling. Gradient masking can make weak attacks look ineffective. Robustness claims need adaptive attacks and independent evaluation; adversarial training usually trades extra compute and sometimes clean accuracy for robustness within the trained threat model.


### **Privacy and Memorization** {#privacy-memorization}

Memorization is not identical to privacy harm, but it can make records distinguishable. Membership inference asks whether a candidate record was used for training, using loss, confidence, gradients, embeddings, or generated content under a defined access model. The [membership-inference study by Shokri et al.](https://arxiv.org/abs/1610.05820) frames privacy leakage as an attack evaluated on members and non-members, not merely a train-test gap.

![A membership inference threat model from candidate record to attack score.](assets/dl21-privacy.svg){fig-align="center" width="72%" fig-alt="A candidate member or non-member is sent through a released model interface, and an attacker converts outputs into a membership guess evaluated by AUROC."}

Differentially private SGD bounds how much the training algorithm's output distribution changes when one record changes. Per-example gradients are clipped to norm (C), averaged, and perturbed:

$$\tilde g=\frac{1}{B}\left(\sum_{i=1}^{B}g_i\min\left(1,\frac{C}{\|g_i\|_2}\right)+\mathcal{N}(0,\sigma^2C^2I)\right).$$

A privacy claim requires a sampling scheme and accountant that reports ((\epsilon,\delta)); clipping and noise alone do not state the guarantee. Privacy also includes data minimization, access control, retention, deletion, secure logs, and output restrictions.

<details>
<summary><strong>PyTorch: audit a loss-threshold membership attack</strong></summary>

```python
audit_size = min(len(test_ids), len(train_ids))
member_x = images[train_ids[:audit_size]]
member_y = targets[train_ids[:audit_size]]
nonmember_x = id_test_x[:audit_size]
nonmember_y = id_test_y[:audit_size]
with torch.inference_mode():
    member_loss = F.cross_entropy(model(member_x), member_y, reduction="none")
    nonmember_loss = F.cross_entropy(model(nonmember_x), nonmember_y, reduction="none")

membership_labels = np.concatenate([np.ones(audit_size), np.zeros(audit_size)])
membership_scores = torch.cat([-member_loss, -nonmember_loss]).numpy()
membership_auc = roc_auc_score(membership_labels, membership_scores)
train_loss_mean = float(member_loss.mean())
test_loss_mean = float(nonmember_loss.mean())
assert 0 <= membership_auc <= 1
print({"attack_AUROC": round(membership_auc, 3), "member_loss": round(train_loss_mean, 3), "nonmember_loss": round(test_loss_mean, 3)})
```

</details>

An AUROC near 0.5 only defeats this attack under this sampling and interface; it is not a proof of privacy. Strong audits match records by class and difficulty, consider multiple attacks, and state auxiliary knowledge. Generative systems additionally test canary exposure, verbatim regurgitation, nearest-neighbor overlap, and extraction under repeated queries.


### **Fairness and Subgroup Evaluation** {#fairness-subgroup-evaluation}

Fairness is a socio-technical property of a system, population, decision, and harm. Metrics such as demographic parity, equalized odds, equal opportunity, predictive parity, and group calibration encode different normative goals and can conflict when base rates differ. The correct question is not “is the model fair?” but which people are affected, what outcome is allocated, which errors cause harm, and what intervention is available.

![A fairness evaluation process grounded in context, groups, metrics, and action.](assets/dl21-fairness.svg){fig-align="center" width="74%" fig-alt="A flow moves from affected context through meaningful groups and metrics to mitigation, while warning that operational slices cannot establish demographic fairness."}

Subgroup evaluation computes metrics for prespecified and intersectional slices, includes sample counts and uncertainty, and reports the worst supported group rather than only an average. Small groups produce unstable estimates; many exploratory slices create multiple-comparison risks. Labels and protected attributes may themselves be missing, noisy, socially constructed, or unsafe to collect.

<details>
<summary><strong>Python: audit class and stroke-density slices without making a demographic claim</strong></summary>

```python
# Thresholds are estimated from training images only.
training_density = images[train_ids].flatten(1).sum(1)
density_thresholds = torch.quantile(training_density, torch.tensor([0.25, 0.75]))
test_density = id_test_x.flatten(1).sum(1)
density_group = torch.bucketize(test_density, density_thresholds)


def grouped_accuracy(prediction, label, group):
    result = {}
    for value in torch.unique(group):
        mask = group == value
        result[int(value)] = {"count": int(mask.sum()), "accuracy": float(prediction[mask].eq(label[mask]).float().mean())}
    return result


density_metrics = grouped_accuracy(id_predictions, id_test_y, density_group)
class_metrics = grouped_accuracy(id_predictions, id_test_y, id_test_y)
worst_density_accuracy = min(item["accuracy"] for item in density_metrics.values())
worst_class_accuracy = min(item["accuracy"] for item in class_metrics.values())
assert sum(item["count"] for item in density_metrics.values()) == len(id_test_y)
print({
    "density_groups": {key: {"n": value["count"], "acc": round(value["accuracy"], 3)} for key, value in density_metrics.items()},
    "worst_density_accuracy": round(worst_density_accuracy, 3),
    "worst_class_accuracy": round(worst_class_accuracy, 3),
})
```

</details>

Stroke density is an observable operational slice, not a protected attribute and not evidence of demographic fairness. UCI Digits lacks the social context needed for that claim. This limitation should appear in the model card and should trigger collection or selection of fit-for-purpose evaluation data before any human-impacting use.


### **Data Provenance and Documentation** {#data-provenance-documentation}

Provenance makes a result traceable to source data, licenses, versions, transformations, split membership, code, configuration, environment, and model artifact. A mutable URL or dataset name is insufficient: upstream content can change while the experiment label remains the same. Hashes identify exact bytes but do not explain collection consent, representativeness, annotation decisions, or known harms.

![A lineage graph from source and snapshot through transformations and model artifact.](assets/dl21-provenance.svg){fig-align="center" width="75%" fig-alt="A sequence links source DOI and license to snapshot checksum, split transformation, code run, and final model artifact with metrics and approvals."}

Datasheets and Data Cards document dataset motivation, composition, collection, preprocessing, uses, distribution, maintenance, and ethical considerations. The [Data Cards work](https://research.google/pubs/data-cards-purposeful-and-transparent-dataset-documentation-for-responsible-ai/) treats documentation as a product for downstream readers, not a form completed after modeling. Lineage systems should be machine-readable enough to reproduce splits and human-readable enough to expose limitations.

<details>
<summary><strong>Python: build a checksum-backed lineage manifest for the chapter experiment</strong></summary>

```python
def sha256_array(array):
    contiguous = np.ascontiguousarray(array)
    return hashlib.sha256(contiguous.tobytes()).hexdigest()


model_buffer = io.BytesIO()
torch.save(model.state_dict(), model_buffer)
lineage_manifest = {
    "dataset": {
        "name": "UCI Optical Recognition of Handwritten Digits",
        "doi": "10.24432/C50P49",
        "license": "CC BY 4.0",
        "feature_hash": sha256_array(digits.data),
        "target_hash": sha256_array(digits.target),
    },
    "split": {
        "unit": "image row",
        "seed": 2121,
        "train_ids_hash": sha256_array(train_ids),
        "validation_ids_hash": sha256_array(val_ids),
        "test_ids_hash": sha256_array(test_ids),
        "ood_definition": "all rows with label 9; excluded from fitting",
    },
    "environment": {"torch": torch.__version__, "sklearn": sklearn.__version__},
    "artifact_sha256": hashlib.sha256(model_buffer.getvalue()).hexdigest(),
}
assert len({lineage_manifest["split"][key] for key in ("train_ids_hash", "validation_ids_hash", "test_ids_hash")}) == 3
assert lineage_manifest["dataset"]["license"] == "CC BY 4.0"
print(json.dumps(lineage_manifest, indent=2)[:900])
```

</details>

The manifest is necessary but not sufficient. A complete record also preserves acquisition and annotation context, data-removal obligations, preprocessing code, evaluator identities, approvals, and incident history. Provenance should survive artifact promotion from notebook to registry to serving environment.


### **Red Teaming and Safety Evaluation** {#red-teaming-safety-evaluation}

Red teaming searches for failures from the perspective of a motivated actor or harmful operating condition. It starts with a threat model: actor, access, knowledge, objective, protected asset, and tolerated impact. Tests then cover malformed inputs, boundary values, distribution shifts, evasion, poisoning, privacy extraction, unsafe capabilities, prompt or tool abuse, and resource exhaustion as relevant to the system.

![Threat-model-driven red teaming and conversion of failures into regression tests.](assets/dl21-red-team.svg){fig-align="center" width="75%" fig-alt="A threat model selects attacks, behavioral oracles judge outcomes, fixes are retested, and each failure enters a regression suite."}

Safety evaluation needs an oracle more specific than “the model should behave well.” Expected behavior may be reject, abstain, cap resources, preserve a policy, request human review, or log an incident. NIST's [AI Risk Management Framework](https://www.nist.gov/itl/ai-risk-management-framework) organizes continuing work around govern, map, measure, and manage; it is a risk-management process, not a claim that one benchmark certifies safety.

<details>
<summary><strong>Python: turn malformed and shifted requests into a regression harness</strong></summary>

```python
def guarded_predict(raw_pixels):
    array = np.asarray(raw_pixels, dtype=np.float32)
    if array.shape not in {(8, 8), (1, 8, 8)}:
        raise ValueError("expected one 8 by 8 image")
    if not np.isfinite(array).all() or array.min() < 0 or array.max() > 16:
        raise ValueError("pixels must be finite values in 0..16")
    tensor = torch.from_numpy(array.reshape(1, 1, 8, 8) / 16.0)
    with torch.inference_mode():
        logits = model(tensor)
        energy = float(-torch.logsumexp(logits, dim=1))
        confidence, prediction = logits.softmax(1).max(1)
    return {"prediction": int(prediction), "confidence": float(confidence), "abstain": energy > ood_energy_threshold}


clean_request = digits.images[int(test_ids[0])]
test_cases = {
    "clean": clean_request,
    "blank": np.zeros((8, 8), dtype=np.float32),
    "inverted": 16.0 - clean_request,
    "malformed": np.zeros((16, 16), dtype=np.float32),
    "out_of_range": np.full((8, 8), 17.0, dtype=np.float32),
}
red_team_results = {}
for name, request in test_cases.items():
    try:
        red_team_results[name] = {"accepted": True, **guarded_predict(request)}
    except ValueError as error:
        red_team_results[name] = {"accepted": False, "reason": str(error)}

assert red_team_results["clean"]["accepted"]
assert not red_team_results["malformed"]["accepted"] and not red_team_results["out_of_range"]["accepted"]
print(red_team_results)
```

</details>

A red-team campaign records coverage and unresolved risk, not only successful attacks. Separate discoverers from fix owners, protect sensitive exploit details, rerun tests after model/runtime changes, and convert every confirmed failure into a versioned regression case. Generative and agentic systems require additional domain-specific tests for content, instruction hierarchy, tool authorization, long-horizon behavior, and human overreliance.


### **Model Cards and Responsible Release** {#model-cards-responsible-release}

A model card accompanies a specific artifact and states what it is, who owns it, intended and out-of-scope uses, training/evaluation data, metrics and slices, limitations, ethical considerations, runtime requirements, and monitoring/rollback expectations. The [Model Cards proposal](https://research.google/pubs/model-cards-for-model-reporting/) emphasizes context and disaggregated performance rather than a leaderboard paragraph.

![Scope, evidence, controls, and release decision in a model card.](assets/dl21-model-card.svg){fig-align="center" width="74%" fig-alt="Three columns document model scope, evaluation evidence, and operational controls before an approved, restricted, or blocked release decision."}

Responsible release is a decision process. Evidence owners define gates before the final test; risk owners accept, mitigate, transfer, or avoid unresolved risk; change management identifies when reevaluation is required. Open release, gated access, API-only access, staged deployment, and non-release are distinct options. Documentation should not convert an unsuitable model into an acceptable one.

<details>
<summary><strong>Python: assemble evidence into a release card with explicit blockers</strong></summary>

```python
release_metrics = {
    "ID_accuracy": baseline_accuracy,
    "calibrated_ECE": calibrated_ece,
    "energy_OOD_AUROC": energy_ood_auc,
    "PGD_accuracy_epsilon_0.16": adversarial_accuracy,
    "membership_attack_AUROC": membership_auc,
    "worst_density_slice_accuracy": worst_density_accuracy,
}
release_gates = {
    "ID_accuracy": release_metrics["ID_accuracy"] >= 0.93,
    "calibration": release_metrics["calibrated_ECE"] <= 0.08,
    "OOD_detection": release_metrics["energy_OOD_AUROC"] >= 0.70,
    "adversarial_robustness": release_metrics["PGD_accuracy_epsilon_0.16"] >= 0.50,
    "membership_audit": release_metrics["membership_attack_AUROC"] <= 0.60,
    "operational_slice": release_metrics["worst_density_slice_accuracy"] >= 0.85,
}
blockers = [name for name, passed in release_gates.items() if not passed]
model_card = {
    "model": "TinyDigitCNN-v1",
    "intended_use": "teaching-scale evaluation of digits 0 through 8",
    "out_of_scope": ["human-impacting decisions", "wild handwriting deployment", "demographic fairness claims"],
    "data": lineage_manifest["dataset"],
    "metrics": {name: round(value, 4) for name, value in release_metrics.items()},
    "limitations": ["digit 9 is the only semantic OOD family", "no demographic attributes", "single seed and small dataset"],
    "release_status": "restricted teaching artifact" if blockers else "approved for intended use",
    "blockers_for_broader_use": blockers,
}
assert model_card["out_of_scope"] and model_card["release_status"]
print(json.dumps(model_card, indent=2))
```

</details>

The release status is intentionally narrow even if every numeric gate passes: those gates cover only this dataset and threat model. A production card adds accountable owners, uncertainty intervals, subgroup definitions, runtime profiles, human-factors evaluation, security review, legal/organizational requirements, and links to incident and rollback procedures.


### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Trustworthiness is not one scalar and interpretability is not one picture. Each method answers a bounded question and carries characteristic failure modes.

| Method | Question answered | Evidence produced | Common overclaim |
|---|---|---|---|
| Task and slice metrics | How often and where does the model succeed? | aggregate, class, subgroup, and selective risk | one average represents all users |
| Calibration / uncertainty | Does confidence support a decision policy? | reliability, proper scores, disagreement | softmax entropy equals epistemic uncertainty |
| OOD detection | Can selected shifts be separated from ID inputs? | AUROC, FPR@TPR, threshold behavior | one OOD dataset proves open-world detection |
| Probe | Is a property decodable from a representation? | held-out controlled readout | decodability proves causal use |
| Saliency / Grad-CAM / IG | Which local features affect a chosen target under a definition? | attribution maps and perturbation tests | a plausible map explains the whole model |
| TCAV / interventions | Is output sensitive to a concept or internal state? | directional tests, ablations, patching | one concept direction identifies a circuit |
| Robustness evaluation | What happens under a specified perturbation/threat model? | corruption curves and adaptive attacks | one norm or attack proves security |
| Privacy audit | Can a stated attacker infer sensitive training influence? | attack advantage under defined access | a failed attack proves privacy |
| Fairness evaluation | How are errors/outcomes distributed in a use context? | supported subgroup metrics and harms | any convenient slice establishes fairness |
| Provenance / cards / red teams | Can evidence, limitations, ownership, and response be traced? | lineage, regression tests, release decision | documentation substitutes for mitigation |

<details>
<summary><strong>Python: verify that every release claim has evidence, a boundary, and an owner</strong></summary>

```python
evidence_registry = [
    {"claim": "ID task quality", "evidence": "test metrics and slices", "boundary": "digits 0-8 fixed split", "owner": "model evaluation"},
    {"claim": "confidence policy", "evidence": "validation temperature and test reliability", "boundary": "current class prior", "owner": "risk evaluation"},
    {"claim": "OOD routing", "evidence": "digit-9 energy AUROC and threshold", "boundary": "one semantic OOD family", "owner": "serving"},
    {"claim": "local explanation", "evidence": "saliency, Grad-CAM, IG, deletion and completeness", "boundary": "selected targets/baselines", "owner": "interpretability"},
    {"claim": "adversarial robustness", "evidence": "PGD under recorded budget", "boundary": "white-box L-inf threat", "owner": "security"},
    {"claim": "privacy risk", "evidence": "loss-threshold membership audit", "boundary": "black-box loss score", "owner": "privacy"},
    {"claim": "responsible release", "evidence": "lineage, model card, red-team regression", "boundary": "teaching use only", "owner": "release authority"},
]
required_fields = {"claim", "evidence", "boundary", "owner"}
assert all(set(record) == required_fields and all(record.values()) for record in evidence_registry)
print({"registered_claims": len(evidence_registry), "all_bounded_and_owned": True})
```

</details>

A defensible workflow defines the decision and harms, freezes data boundaries, evaluates aggregate and slices, calibrates uncertainty, tests shifts and attacks, investigates mechanisms with controlled methods, audits privacy and fairness under explicit limitations, records provenance, and links failures to owners and rollback. Evidence must be updated whenever data, model, runtime, policy, or use context changes.
